<a href="https://colab.research.google.com/github/Devansh152005/ILLM_LAB/blob/main/LLM_LAB1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
import re
import random
from collections import defaultdict
from datasets import load_dataset

# Updated file paths including the 'data/' directory prefix and accurate filenames
INDIC_LANGUAGES = [
    "data/as.txt", "data/bd.txt", "data/bn.txt", "data/dg.txt", "data/en.txt",
    "data/gom.txt", "data/gu.txt", "data/hi-1.txt", "data/hi-2.txt", "data/hi-3.txt",
    "data/kn.txt", "data/ml.txt", "data/mr.txt", "data/ne.txt", "data/or.txt",
    "data/pa.txt", "data/sa.txt", "data/sd.txt", "data/ta.txt", "data/te.txt",
    "data/ur.txt"
]

def sentence_tokenize(text):
    """
    Splits a paragraph into sentences using common sentence terminators.
    Handles Indic purna virama (|), double purna virama (||), and standard punctuation.
    """
    sentences = re.split(r'[।॥!?.|]\s*', text)
    return [s.strip() for s in sentences if len(s.strip()) > 10]

def create_dataset(samples_per_lang=1000):
    dataset = []
    print("Loading datasets and extracting sentences...")

    for lang_path in INDIC_LANGUAGES:
        # Extract clean label (e.g., 'data/hi-1.txt' -> 'hi-1')
        lang_label = lang_path.split('/')[-1].replace('.txt', '')
        try:
            # Pass data_files directly pointing to the 'data/' folder path
            ds = load_dataset(
                "ai4bharat/IndicCorpV2",
                data_files={"train": lang_path},
                split="train",
                streaming=True
            )
            lang_sentences = []

            for item in ds:
                text = item.get("text", "")
                sents = sentence_tokenize(text)
                for s in sents:
                    lang_sentences.append((s, lang_label))
                    if len(lang_sentences) >= samples_per_lang:
                        break
                if len(lang_sentences) >= samples_per_lang:
                    break

            dataset.extend(lang_sentences)
            print(f"Collected {len(lang_sentences)} sentences for {lang_label}")
        except Exception as e:
            print(f"Skipping {lang_label}: {e}")

    return dataset

def stratified_split(dataset, train_ratio=0.8, val_ratio=0.1):
    grouped = defaultdict(list)
    for text, label in dataset:
        grouped[label].append(text)

    train_data, val_data, test_data = [], [], []

    # Set seed for reproducibility
    random.seed(42)

    for label, texts in grouped.items():
        random.shuffle(texts)
        n = len(texts)
        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)

        train_data.extend([(t, label) for t in texts[:n_train]])
        val_data.extend([(t, label) for t in texts[n_train:n_train + n_val]])
        test_data.extend([(t, label) for t in texts[n_train + n_val:]])

    return train_data, val_data, test_data

In [34]:
import numpy as np

class CustomLogisticRegression:
    def __init__(self, lr=0.1, epochs=20):
        self.lr = lr
        self.epochs = epochs
        self.weights = None # Matrix: [num_features, num_classes]
        self.biases = None  # Vector: [num_classes]
        self.classes = []
        self.class_to_idx = {}

    def _softmax(self, z):
        # z shape: (num_samples, num_classes)
        # Subtract max for numerical stability
        shift_z = z - np.max(z, axis=1, keepdims=True)
        exps = np.exp(shift_z)
        return exps / np.sum(exps, axis=1, keepdims=True)

    def fit(self, X, y, num_features):
        # Ensure X is a numpy array for vectorized operations
        X = np.array(X)
        self.classes = sorted(list(set(y)))
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        num_classes = len(self.classes)

        # Initialize weights and biases
        self.weights = np.zeros((num_features, num_classes))
        self.biases = np.zeros(num_classes)

        # One-hot encode targets
        y_indices = np.array([self.class_to_idx[label] for label in y])
        y_one_hot = np.zeros((len(y), num_classes))
        y_one_hot[np.arange(len(y)), y_indices] = 1

        print(f"Training Softmax Regression for {num_classes} classes...")
        for epoch in range(self.epochs):
            # Forward pass: (N, F) dot (F, K) + (K,)
            scores = np.dot(X, self.weights) + self.biases
            probs = self._softmax(scores)

            # Gradient Calculation
            error = probs - y_one_hot
            dw = np.dot(X.T, error) / len(y)
            db = np.sum(error, axis=0) / len(y)

            # Update parameters
            self.weights -= self.lr * dw
            self.biases -= self.lr * db

            if (epoch + 1) % 5 == 0:
                print(f"Epoch {epoch+1}/{self.epochs} completed")

    def predict(self, X):
        X = np.array(X)
        scores = np.dot(X, self.weights) + self.biases
        probs = self._softmax(scores)
        indices = np.argmax(probs, axis=1)
        return [self.classes[i] for i in indices]

In [35]:
def compute_macro_f1(y_true, y_pred):
    classes = sorted(list(set(y_true)))
    f1_scores = []

    for c in classes:
        tp = sum(1 for yt, yp in zip(y_true, y_pred) if yt == c and yp == c)
        fp = sum(1 for yt, yp in zip(y_true, y_pred) if yt != c and yp == c)
        fn = sum(1 for yt, yp in zip(y_true, y_pred) if yt == c and yp != c)

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

        if precision + recall == 0:
            f1 = 0.0
        else:
            f1 = 2 * (precision * recall) / (precision + recall)

        f1_scores.append(f1)

    # Calculate unweighted arithmetic mean (Macro-F1)
    macro_f1 = sum(f1_scores) / len(f1_scores)
    return macro_f1

In [37]:
import math
import numpy as np
from collections import defaultdict, Counter

# Redefining the class here to force the kernel to use the correct version
class CustomTFIDFVectorizer:
    def __init__(self, max_features=15000, tf_type='tf0', idf_type='idf0'):
        self.max_features = max_features
        self.tf_type = tf_type
        self.idf_type = idf_type
        self.vocabulary = {}
        self.df_counts = defaultdict(int)
        self.total_docs = 0
        self.max_df = 0
        self.idf = None

    def _extract_ngrams(self, text):
        ngrams = []
        words = text.split()
        ngrams.extend(words)
        for i in range(len(words) - 1):
            ngrams.append(words[i] + " " + words[i+1])
        for n in (2, 3, 4):
            for i in range(len(text) - n + 1):
                ngrams.append(text[i:i+n])
        return ngrams

    def fit(self, raw_documents):
        self.total_docs = len(raw_documents)
        ngram_freqs = defaultdict(int)
        for doc in raw_documents:
            ngrams = self._extract_ngrams(doc)
            for ng in set(ngrams): self.df_counts[ng] += 1
            for ng in ngrams: ngram_freqs[ng] += 1
        sorted_ngrams = sorted(ngram_freqs.items(), key=lambda x: x[1], reverse=True)[:self.max_features]
        self.vocabulary = {ng: idx for idx, (ng, _) in enumerate(sorted_ngrams)}
        if self.df_counts: self.max_df = max(self.df_counts.values())
        self.idf = np.zeros(len(self.vocabulary), dtype=np.float32)
        for ng, idx in self.vocabulary.items():
            df = self.df_counts[ng]
            if self.idf_type == 'idf0':
                self.idf[idx] = math.log((1 + self.total_docs) / (1 + df)) + 1.0
            elif self.idf_type == 'idf1':
                self.idf[idx] = math.log((1 + self.max_df) / (1 + df)) + 1.0

    def transform(self, raw_documents):
        rows, cols = len(raw_documents), len(self.vocabulary)
        X = np.zeros((rows, cols), dtype=np.float32)
        for row_idx, doc in enumerate(raw_documents):
            ngrams = self._extract_ngrams(doc)
            counts = Counter(ngrams)
            total_ngrams = len(ngrams) or 1
            max_freq = max(counts.values()) if counts else 1
            for ng, count in counts.items():
                if ng in self.vocabulary:
                    col_idx = self.vocabulary[ng]
                    if self.tf_type == 'tf0': tf = float(count)
                    elif self.tf_type == 'tf1': tf = count / total_ngrams
                    elif self.tf_type == 'tf2': tf = 0.5 + 0.5 * (count / max_freq)
                    X[row_idx, col_idx] = tf * self.idf[col_idx]
            norm = np.linalg.norm(X[row_idx])
            if norm > 0: X[row_idx] /= norm
        return X

if __name__ == "__main__":
    # 1. Prepare raw dataset
    data = create_dataset(samples_per_lang=1200)
    train_set, val_set, test_set = stratified_split(data, train_ratio=0.8, val_ratio=0.1)
    X_train_raw, y_train = zip(*train_set)
    X_val_raw, y_val = zip(*val_set)
    X_test_raw, y_test = zip(*test_set)

    # 2. Extract Features using the version defined above
    print("Fitting TF-IDF Vectorizer (Softmax Compatible)... ")
    vectorizer = CustomTFIDFVectorizer(max_features=15000, tf_type='tf1', idf_type='idf0')
    vectorizer.fit(X_train_raw)

    print("Transforming to dense matrices...")
    X_train = vectorizer.transform(X_train_raw)
    X_val = vectorizer.transform(X_val_raw)
    X_test = vectorizer.transform(X_test_raw)

    # 3. Train Softmax Regression
    classifier = CustomLogisticRegression(lr=0.5, epochs=15)
    classifier.fit(X_train, y_train, num_features=X_train.shape[1])

    # 4. Evaluation
    val_preds = classifier.predict(X_val)
    print(f"\n--- Validation Macro-F1: {compute_macro_f1(y_val, val_preds):.4f} ---")
    test_preds = classifier.predict(X_test)
    print(f"--- Test Macro-F1: {compute_macro_f1(y_test, test_preds):.4f} ---")

Loading datasets and extracting sentences...
Collected 1200 sentences for as
Collected 1200 sentences for bd
Collected 1200 sentences for bn
Collected 1200 sentences for dg
Collected 1200 sentences for en
Collected 1200 sentences for gom
Collected 1200 sentences for gu
Collected 1200 sentences for hi-1
Collected 1200 sentences for hi-2
Skipping hi-3: 'utf-8' codec can't decode byte 0xb9 in position 0: invalid start byte
Collected 1200 sentences for kn
Collected 1200 sentences for ml
Collected 1200 sentences for mr
Collected 1200 sentences for ne
Collected 1200 sentences for or
Collected 1200 sentences for pa
Collected 1200 sentences for sa
Collected 1200 sentences for sd
Collected 1200 sentences for ta
Collected 1200 sentences for te
Collected 1200 sentences for ur
Fitting TF-IDF Vectorizer (Softmax Compatible)... 
Transforming to dense matrices...
Training Softmax Regression for 20 classes...
Epoch 5/15 completed
Epoch 10/15 completed
Epoch 15/15 completed

--- Validation Macro-F1: 0.

NOW the assignment 2 starts

In [38]:
import math
import numpy as np
from collections import defaultdict, Counter

class CustomTFIDFVectorizer:
    def __init__(self, max_features=15000, tf_type='tf0', idf_type='idf0'):
        """
        tf_type:
          'tf0' : Raw count f_{t,d}
          'tf1' : Normalized by total ngrams in sentence (f_{t,d} / N_d)
          'tf2' : Normalized by max ngram frequency in sentence (0.5 + 0.5 * (f_{t,d} / max_f))

        idf_type:
          'idf0' : Standard Smoothed IDF -> log((1 + D) / (1 + DF)) + 1
          'idf1' : Max DF Normalized IDF -> log((1 + max_DF) / (1 + DF)) + 1
        """
        self.max_features = max_features
        self.tf_type = tf_type
        self.idf_type = idf_type
        self.vocabulary = {}
        self.df_counts = defaultdict(int)
        self.total_docs = 0
        self.max_df = 0
        self.idf = None

    def _extract_ngrams(self, text):
        ngrams = []
        words = text.split()

        # Word Unigrams & Bigrams
        ngrams.extend(words)
        for i in range(len(words) - 1):
            ngrams.append(words[i] + " " + words[i+1])

        # Character 2, 3, 4-grams
        for n in (2, 3, 4):
            for i in range(len(text) - n + 1):
                ngrams.append(text[i:i+n])

        return ngrams

    def fit(self, raw_documents):
        self.total_docs = len(raw_documents)
        ngram_freqs = defaultdict(int)

        for doc in raw_documents:
            ngrams = self._extract_ngrams(doc)
            unique_ngrams = set(ngrams)
            for ng in unique_ngrams:
                self.df_counts[ng] += 1
            for ng in ngrams:
                ngram_freqs[ng] += 1

        # Select top features
        sorted_ngrams = sorted(ngram_freqs.items(), key=lambda x: x[1], reverse=True)[:self.max_features]
        self.vocabulary = {ng: idx for idx, (ng, _) in enumerate(sorted_ngrams)}

        if self.df_counts:
            self.max_df = max(self.df_counts.values())

        # Compute IDF
        self.idf = np.zeros(len(self.vocabulary), dtype=np.float32)
        for ng, idx in self.vocabulary.items():
            df = self.df_counts[ng]
            if self.idf_type == 'idf0':
                self.idf[idx] = math.log((1 + self.total_docs) / (1 + df)) + 1.0
            elif self.idf_type == 'idf1':
                self.idf[idx] = math.log((1 + self.max_df) / (1 + df)) + 1.0

    def transform(self, raw_documents):
        rows = len(raw_documents)
        cols = len(self.vocabulary)
        X = np.zeros((rows, cols), dtype=np.float32)

        for row_idx, doc in enumerate(raw_documents):
            ngrams = self._extract_ngrams(doc)
            counts = Counter(ngrams)
            total_ngrams = len(ngrams) or 1
            max_freq = max(counts.values()) if counts else 1

            for ng, count in counts.items():
                if ng in self.vocabulary:
                    col_idx = self.vocabulary[ng]

                    if self.tf_type == 'tf0':
                        tf = float(count)
                    elif self.tf_type == 'tf1':
                        tf = count / total_ngrams
                    elif self.tf_type == 'tf2':
                        tf = 0.5 + 0.5 * (count / max_freq)

                    X[row_idx, col_idx] = tf * self.idf[col_idx]

            # L2 Row Normalization
            norm = np.linalg.norm(X[row_idx])
            if norm > 0:
                X[row_idx] /= norm

        return X

In [39]:
if __name__ == "__main__":
    # Assuming X_train_raw, X_val_raw, X_test_raw, y_train, y_val, y_test are already created

    combinations = [
        ("TF0 (Raw Count)", "IDF0 (Standard)", "tf0", "idf0"),
        ("TF0 (Raw Count)", "IDF1 (Max DF Norm)", "tf0", "idf1"),
        ("TF1 (Sentence Length Norm)", "IDF0 (Standard)", "tf1", "idf0"),
        ("TF1 (Sentence Length Norm)", "IDF1 (Max DF Norm)", "tf1", "idf1"),
        ("TF2 (Max Freq Norm)", "IDF0 (Standard)", "tf2", "idf0"),
        ("TF2 (Max Freq Norm)", "IDF1 (Max DF Norm)", "tf2", "idf1"),
    ]

    print("\n" + "="*85)
    print(f"{'TF Strategy':<28} | {'IDF Strategy':<22} | {'Val Macro-F1':<12} | {'Test Macro-F1':<12}")
    print("="*85)

    for tf_label, idf_label, tf_type, idf_type in combinations:
        vec = CustomTFIDFVectorizer(max_features=15000, tf_type=tf_type, idf_type=idf_type)
        vec.fit(X_train_raw)

        X_tr = vec.transform(X_train_raw)
        X_va = vec.transform(X_val_raw)
        X_te = vec.transform(X_test_raw)

        # Uses your CustomLogisticRegression class with len(vec.vocabulary)
        model = CustomLogisticRegression(lr=0.5, epochs=15)
        model.fit(X_tr, y_train, num_features=len(vec.vocabulary))

        val_preds = model.predict(X_va)
        test_preds = model.predict(X_te)

        val_f1 = compute_macro_f1(y_val, val_preds)
        test_f1 = compute_macro_f1(y_test, test_preds)

        print(f"{tf_label:<28} | {idf_label:<22} | {val_f1:<12.4f} | {test_f1:<12.4f}")

    print("="*85)


TF Strategy                  | IDF Strategy           | Val Macro-F1 | Test Macro-F1
Training Softmax Regression for 20 classes...
Epoch 5/15 completed
Epoch 10/15 completed
Epoch 15/15 completed
TF0 (Raw Count)              | IDF0 (Standard)        | 0.8531       | 0.8558      
Training Softmax Regression for 20 classes...
Epoch 5/15 completed
Epoch 10/15 completed
Epoch 15/15 completed
TF0 (Raw Count)              | IDF1 (Max DF Norm)     | 0.8620       | 0.8651      
Training Softmax Regression for 20 classes...
Epoch 5/15 completed
Epoch 10/15 completed
Epoch 15/15 completed
TF1 (Sentence Length Norm)   | IDF0 (Standard)        | 0.8531       | 0.8558      
Training Softmax Regression for 20 classes...
Epoch 5/15 completed
Epoch 10/15 completed
Epoch 15/15 completed
TF1 (Sentence Length Norm)   | IDF1 (Max DF Norm)     | 0.8620       | 0.8651      
Training Softmax Regression for 20 classes...
Epoch 5/15 completed
Epoch 10/15 completed
Epoch 15/15 completed
TF2 (Max Freq Norm)    